# Smart Classroom: train the gesture models on a GPU (Colab / Kaggle)

1. **Runtime → Change runtime type → T4 GPU**
2. On your PC: `python -m smartclass.training.pack` → creates `SmartClassroom_train.zip`
3. Run the cells below in order (cell 1 asks you to upload the zip)
4. Download `SmartClassroom_results.zip` and unzip it **into** your `SmartClassroom` folder

The code is the same as in the app (`smartclass/training/train_gestures.py`), so results are reproducible (seed 42).

In [ ]:
# 1. Upload the project zip (or put it in Google Drive and copy it here)
import os, zipfile
if not os.path.exists('SmartClassroom_train.zip'):
    from google.colab import files          # on Kaggle: add the zip as a Dataset instead
    files.upload()
zipfile.ZipFile('SmartClassroom_train.zip').extractall('SmartClassroom')
%cd SmartClassroom
!ls data/gestures

In [ ]:
# 2. Check the GPU (should print a Tesla T4 or similar)
import torch
print(torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: change the runtime type!')

In [ ]:
# 3. Train all models (MLP baseline, LSTM main, GRU, ablation, CNN+GRU)
#    Fewer models:  --models lstm mlp      More epochs:  --epochs 100
!python -m smartclass.training.train_gestures --epochs 80

In [ ]:
# 4. Results table + figures
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('output/gestures/results.csv'))
print(open('output/gestures/split.txt').read())
display(Image('output/gestures/comparison.png'))
for name in ['lstm', 'mlp']:
    display(Image(f'output/gestures/{name}_curves.png'))
    display(Image(f'output/gestures/{name}_confusion.png'))

In [ ]:
# 5. Download the trained models + figures. Unzip INTO the SmartClassroom folder on your PC.
!zip -qr ../SmartClassroom_results.zip models/gesture_*.pt output/gestures
try:
    from google.colab import files
    files.download('../SmartClassroom_results.zip')
except ImportError:
    print('Kaggle: get SmartClassroom_results.zip from the Output panel')